# Pairwise distance scatter

Each dot is a target. Color by type (NC / PC / targeting). The headline pair is **Hon CM vs Gersbach Hep**; an all-pairs grid follows.

**Input:** `results/pairwise_distance_scatter/per_target_long.tsv` (from `scripts/build_long_per_target_table.py`)
**Output:** `results/pairwise_distance_scatter/distance_scatter_{HonCM_vs_GersbachHep,all_pairs}.pdf`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

HERE = Path.cwd()
EDIST = HERE if HERE.name == "edist" else HERE.parent
JAMB = EDIST.parents[2]
REPO = JAMB.parents[2]
RESULTS = EDIST / "results" / "pairwise_distance_scatter"
RESULTS.mkdir(parents=True, exist_ok=True)

with open(REPO / "config/colors/production_TF-Perturb-seq.yaml") as f:
    CFG = yaml.safe_load(f)

SHORT = {
    "Hon_WTC11-cardiomyocyte-differentiation_TF-Perturb-seq": "HonCM",
    "Huangfu_HUES8-definitive-endoderm-differentiation_TF-Perturb-seq": "HuangfuDE",
    "Huangfu_HUES8-embryonic-stemcell-differentiation_TF-Perturb-seq": "HuangfuESC",
    "Gersbach_WTC11-hepatocyte-differentiation_TF-Perturb-seq": "GersbachHep",
    "Engreitz_WTC11-endothelial-cells_TF-Perturb-seq": "EngreitzEndo",
}
SHORT_TO_FULL = {v: k for k, v in SHORT.items()}
DATASET_ORDER = ["HonCM", "HuangfuDE", "HuangfuESC", "GersbachHep"]
COLORS = {s: CFG["dataset_colors"][SHORT_TO_FULL[s]] for s in DATASET_ORDER}

print("RESULTS:", RESULTS)


In [ ]:
long = pd.read_csv(RESULTS / "per_target_long.tsv", sep="\t")

def scatter_pair(x_short: str, y_short: str, ax) -> None:
    px = long[long["dataset_short"] == x_short].set_index("target_id")["distance_mean"]
    py = long[long["dataset_short"] == y_short].set_index("target_id")["distance_mean"]
    typ = long[long["dataset_short"] == x_short].set_index("target_id")["type"]
    common = px.index.intersection(py.index)
    px, py, typ = px.loc[common], py.loc[common], typ.loc[common]
    type_colors = {"targeting": "#888888", "negative control": "#E07B30", "positive control": "#3B6FB6"}
    for t, c in type_colors.items():
        m = typ == t
        ax.scatter(px[m], py[m], s=10, c=c, alpha=0.6, edgecolors="none", label=t)
    lo = min(px.min(), py.min())
    hi = max(px.max(), py.max())
    ax.plot([lo, hi], [lo, hi], color="black", linewidth=0.6, linestyle="--")
    ax.set_xlabel(f"{x_short} distance_mean")
    ax.set_ylabel(f"{y_short} distance_mean")
    ax.spines[["top", "right"]].set_visible(False)
    m = typ == "targeting"
    r = np.corrcoef(px[m], py[m])[0, 1]
    ax.text(0.04, 0.95, f"r = {r:.2f}\nn = {m.sum()}", transform=ax.transAxes,
            va="top", fontsize=9)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
scatter_pair("HonCM", "GersbachHep", ax)
ax.set_title("Mean-distance scatter: Hon CM vs Gersbach Hep")
ax.legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
fig.savefig(RESULTS / "distance_scatter_HonCM_vs_GersbachHep.pdf")
plt.show()

n = len(DATASET_ORDER)
fig, axes = plt.subplots(n, n, figsize=(3.0 * n, 3.0 * n))
for i, ds_y in enumerate(DATASET_ORDER):
    for j, ds_x in enumerate(DATASET_ORDER):
        ax = axes[i, j]
        if i == j:
            ax.text(0.5, 0.5, ds_x, ha="center", va="center", fontsize=11,
                    transform=ax.transAxes, fontweight="bold")
            ax.set_xticks([]); ax.set_yticks([])
            ax.spines[["top", "right", "left", "bottom"]].set_visible(False)
        elif i > j:
            scatter_pair(ds_x, ds_y, ax)
            ax.set_title("")
            if i != n - 1: ax.set_xlabel("")
            if j != 0: ax.set_ylabel("")
        else:
            ax.set_visible(False)
plt.tight_layout()
fig.savefig(RESULTS / "distance_scatter_all_pairs.pdf")
plt.show()